In [1]:
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import optuna
from IPython.display import HTML, Markdown
import xarray as xr
import plotly.graph_objects as go
import seaborn as sns

import nn_gmm as nng
import ml_tools as mlt

In [2]:
hp_result_dir = Path("/Users/claudy/dev/work/data/nn_gmm/hp_opt/1104_1103")
db_ffp = hp_result_dir / "1104_1103.db"

In [3]:
study_id = hp_result_dir.name

study = optuna.load_study(
    storage=f"sqlite:///{hp_result_dir}/{study_id}.db"
    , study_name=study_id
)

In [4]:
df = study.trials_dataframe()
df = df.loc[df.state == "COMPLETE"]
df["trial_id"] = [f"trial_{row['number']:03d}" for _, row in df.iterrows()]
df = df.sort_values("value")

In [5]:
result_cols = ["number",'value', 'duration',
       'params_activation_fn', 'params_batch_size', 'params_dropout_rate',
       'params_l2_reg', 'params_learning_rate', 'params_n_layers',
       'params_unit_size', 'params_use_batch_norm']
param_cols = [col for col in df.columns if col.startswith("params_")]

In [6]:
HTML(df[result_cols].head(25).to_html())

,number,value,duration,params_activation_fn,params_batch_size,params_dropout_rate,params_l2_reg,params_learning_rate,params_n_layers,params_unit_size,params_use_batch_norm
51,51,-1.702466,0 days 00:17:07.546502,leaky_relu,1024,0.0,0.0010,0.010,4,32,False
49,49,-1.699927,0 days 00:17:01.601511,leaky_relu,1024,0.0,0.0010,0.010,4,32,False
53,53,-1.698510,0 days 00:17:02.044751,leaky_relu,1024,0.0,0.0010,0.010,4,32,False
75,75,-1.692180,0 days 00:17:50.104500,relu,1024,0.0,0.0010,0.010,5,32,False
71,71,-1.688063,0 days 00:18:02.787142,relu,1024,0.0,0.0010,0.010,5,32,False
73,73,-1.685678,0 days 00:18:06.342929,relu,1024,0.0,0.0010,0.010,5,32,False
57,57,-1.685451,0 days 00:18:41.684400,elu,1024,0.0,0.0010,0.005,4,32,True
74,74,-1.682367,0 days 00:17:43.152464,relu,1024,0.0,0.0010,0.010,5,32,False
72,72,-1.681942,0 days 00:18:48.686856,relu,1024,0.0,0.0010,0.010,5,32,False
92,92,-1.680142,0 days 00:17:44.277895,relu,1024,0.0,0.0010,0.010,5,32,False


In [7]:
# Group by parameters and get mean value to address duplicates
Markdown(df.groupby(param_cols)["value"].mean().sort_values().head(10).to_frame().to_markdown())

|                                                      |    value |
|:-----------------------------------------------------|---------:|
| ('leaky_relu', 1024, 0.0, 0.001, 0.01, 4, 32, False) | -1.68164 |
| ('leaky_relu', 4096, 0.0, 0.005, 0.01, 5, 64, False) | -1.67251 |
| ('relu', 512, 0.0, 0.001, 0.01, 5, 32, False)        | -1.67039 |
| ('relu', 1024, 0.0, 0.001, 0.01, 5, 32, False)       | -1.6698  |
| ('elu', 1024, 0.0, 0.001, 0.005, 4, 32, True)        | -1.66765 |
| ('leaky_relu', 1024, 0.0, 0.001, 0.01, 3, 32, False) | -1.66419 |
| ('relu', 512, 0.0, 0.0005, 0.005, 5, 32, False)      | -1.66286 |
| ('leaky_relu', 4096, 0.0, 0.005, 0.01, 3, 64, False) | -1.66002 |
| ('leaky_relu', 4096, 0.0, 0.005, 0.01, 4, 64, False) | -1.65568 |
| ('relu', 4096, 0.0, 0.005, 0.01, 3, 64, False)       | -1.63951 |

## Loss Investigation


In [8]:
# Load the metrics for every trial
trial_dirs = list(hp_result_dir.glob("trial_*"))
trial_names = [cur_dir.name for cur_dir in trial_dirs]

In [9]:
metrics_list = [xr.load_dataarray(cur_dir / "metrics.nc") for cur_dir in trial_dirs]

In [ ]:
metrics_da = xr.concat(metrics_list, dim="trial")
metrics_da = metrics_da.assign_coords({"trial": trial_names})

In [11]:
n_trials = 15

top_x_trials = df.head(n_trials)["trial_id"].values
# line_objects = [go.Scatter(x=metrics_da.coords["epoch"].values, y=metrics_da.sel(trial=trial, metric="w_loss_hist_val").median(dim="cv_iter"), mode="lines", name=trial) for trial in top_10_trials]
# fig = go.Figure(data=line_objects)

fig = go.Figure()
fig.update_layout(width=1600, height=800,
                  margin=dict(l=10, r=10, t=30, b=10)  # left, right, top, bottom
)
fig.update_yaxes(range=[-1.8, -0.8]) 

colors = sns.color_palette("tab10", n_colors=n_trials).as_hex()
for cur_trial, cur_color in zip(top_x_trials, colors):
    cur_16th = metrics_da.sel(trial=cur_trial, metric="w_loss_hist_val").quantile(0.16, dim="cv_iter")
    cur_84th = metrics_da.sel(trial=cur_trial, metric="w_loss_hist_val").quantile(0.84, dim="cv_iter")
    cur_std = metrics_da.sel(trial=cur_trial, metric="w_loss_hist_val").std(dim="cv_iter")
    

    fig.add_trace(go.Scatter(x=metrics_da.coords["epoch"].values, y=metrics_da.sel(trial=cur_trial, metric="w_loss_hist_val").median(dim="cv_iter"), mode="lines", 
                             name=cur_trial, legendgroup=cur_trial, line=dict(color=cur_color)))
    fig.add_trace(go.Scatter(x=metrics_da.coords["epoch"].values, y=cur_16th, mode="lines", line=dict(dash="dash", color=cur_color), showlegend=False, legendgroup=cur_trial))
    fig.add_trace(go.Scatter(x=metrics_da.coords["epoch"].values, y=cur_84th, mode="lines", line=dict(dash="dash", color=cur_color), showlegend=False, legendgroup=cur_trial))

fig.show()